# Bayes Classifier

In [1]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import os
from sklearn import mixture


#Function to evaluate a gaussian PDF (probability density function) acourding to the mean and covariance provided as input
def pdfGauss(X, mean, cov):
    '''
    X: input data points for which we want to evaluate the Gaussian PDF.
    mean: the mean vector of the Gaussian distribution.
    cov: the covariance matrix of the Gaussian distribution.
    '''
    covInv = np.linalg.inv(cov)
    dim = cov.shape[0]
    val = np.array([])
    for i in range(X.shape[0]):
        dist = ((np.array([X[i,:]-mean]))@covInv@(np.array([X[i,:]-mean])).T).squeeze()
        val = np.append(val, np.exp(-0.5*dist)/((2*np.pi)**(dim/2)*np.linalg.det(cov)**0.5))
    return np.array([val]).T


#Fit bayes for the training data
def fitBayes(Xtr, ytr):
    ix1 = np.where(ytr==1)[0]
    ix2 = np.where(ytr==2)[0]

    #Compute priors
    Pw1 = ix1.shape[0]/(ix1.shape[0]+ix2.shape[0])
    Pw2 = ix2.shape[0]/(ix1.shape[0]+ix2.shape[0])

    # Estimate gaussian conditional PDFs --> fit two Gaussian Mixture Models with 1 component
    clf1 = mixture.GaussianMixture(n_components=1)
    clf2 = mixture.GaussianMixture(n_components=1)
    mod1 = clf1.fit(Xtr[ix1,:])
    mod2 = clf2.fit(Xtr[ix2,:])

    return {'mean1': mod1.means_.squeeze(), 'mean2': mod2.means_.squeeze(),
            'cov1': mod1.covariances_[0], 'cov2': mod2.covariances_[0],
            'Pw1': Pw1, 'Pw2': Pw2}


#Apply a trained bayes classifier - give the model output
def useBayes(Xte, model):
    '''
    Xte: input data points for which we want to predict the class labels.
    model: dictionary with the means, covariances and priors of each class.
    '''
    Pw1X = pdfGauss(Xte, model['mean1'], model['cov1'])*model['Pw1']
    Pw2X = pdfGauss(Xte, model['mean2'], model['cov2'])*model['Pw2']

    # Returns 1 where Pw1X > Pw2X and 2 otherwise
    return ((-np.sign(Pw1X-Pw2X))*0.5+1.5).squeeze()


def plotDecision(X, y, model, granularity=0.01, classContours=False, f1name="f1", f2name="f2"):
    #Identify class indexes
    ix1 = np.where(y==1)[0]
    ix2 = np.where(y==2)[0]

    #Create vectors for plotting
    xRange = np.arange(X[:,0].min(), X[:,0].max()+0.5, granularity)
    yRange = np.arange(X[:,1].min(), X[:,1].max(), granularity)

    Xp, Yp = np.meshgrid(xRange, yRange)
    XX = np.array([Xp.ravel(), Yp.ravel()]).T

    #Sample class one pdf, just to get contours
    Z1 = pdfGauss(XX, model['mean1'], model['cov1']).reshape(Xp.shape)
    #Sample class two pdf, just to get contours
    Z2 = pdfGauss(XX, model['mean2'], model['cov2']).reshape(Xp.shape)

    #Plot samples
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=X[ix1,0], y=X[ix1,1], name='1'))
    fig.add_trace(go.Scatter(x=X[ix2,0], y=X[ix2,1], name='2'))
    fig.update_traces(marker=dict(size=8))

    #Plot means
    fig.add_trace(go.Scatter(x=np.array([model['mean1'][0]]), y=np.array([model['mean1'][1]]),
                             marker_size=20, marker_symbol='x', name='1 mean',
                             marker=dict(color="blue", line=dict(width=1, color='black'))))
    fig.add_trace(go.Scatter(x=np.array([model['mean2'][0]]), y=np.array([model['mean2'][1]]),
                             marker_size=20, marker_symbol='x', name='2 mean',
                             marker=dict(color="red", line=dict(width=1, color='black'))))
    fig.update_traces(mode='markers')

    #Compute Bayes decision for the grid
    yteP = np.zeros((yRange.shape[0], xRange.shape[0]))
    for i in range(xRange.shape[0]):
        for j in range(yRange.shape[0]):
            yteP[j,i] = useBayes(np.array([[xRange[i], yRange[j]]]), model)

    if classContours:
        fig.add_trace(go.Contour(z=Z1, x=xRange, y=yRange,
                                 contours_coloring='lines', line_width=1, showscale=False))
        fig.add_trace(go.Contour(z=Z2, x=xRange, y=yRange,
                                 contours_coloring='lines', line_width=1, showscale=False))

    #Add decision contour
    fig.add_trace(go.Contour(z=yteP, x=xRange, y=yRange, contours_coloring='lines',
                             name='Decision Boundary', colorscale='Greys',
                             line_width=4, showscale=False))
    fig.update_traces(ncontours=1, selector=dict(type='contour'))
    fig.update_xaxes(title_text=f1name)
    fig.update_yaxes(title_text=f2name)
    fig.update_layout(autosize=False, width=900, height=800)
    fig.show()


#--------------------------------------------------------------------------------------------
# Ler o ficheiro CSV: separador ';', decimal ',' e saltar a linha vazia (linha 2)
df = pd.read_csv(
    r"C:/Users/josem/OneDrive/Ambiente de Trabalho/Zé Miguel/UNIVERSIDADE/5ANO/Rodrigo_Jose/AC/datasets/CORK_STOPPERS.csv",
    sep=";",
    decimal=",",
    skiprows=[1]
)

# Selecionar as colunas pretendidas e remover linhas com valores em falta
CorkStopp = df[["C", "ART", "N.1", "PRT", "ARM", "PRM",
                "ARTG", "NG", "PRTG", "RAAR", "RAN"]].dropna().copy()

fnames = CorkStopp.columns[1:]

# Índices posicionais das amostras de cada classe (classes 1 e 2 para treino)
ixSuper = np.where(CorkStopp["C"].to_numpy() == 1)[0]
ixAverage = np.where(CorkStopp["C"].to_numpy() == 2)[0]

print("Dimensões do dataset:", CorkStopp.shape)
print("Features:", list(fnames))
print("Número de amostras da classe 1:", len(ixSuper))
print("Número de amostras da classe 2:", len(ixAverage))

#Extract features for training
ARTSup = CorkStopp['ART'].to_numpy()[ixSuper]
ARTAvg = CorkStopp['ART'].to_numpy()[ixAverage]

PRTSup = CorkStopp['PRT'].to_numpy()[ixSuper]
PRTAvg = CorkStopp['PRT'].to_numpy()[ixAverage]

X1 = np.array([ARTSup, PRTSup]).T
X2 = np.array([ARTAvg, PRTAvg]).T
XtrC = np.concatenate((X1, X2), axis=0)
ytrC = np.concatenate((CorkStopp['C'].to_numpy()[ixSuper],
                       CorkStopp['C'].to_numpy()[ixAverage]))

#Fit bayes to training
modelC = fitBayes(XtrC, ytrC)

#Evaluate on the training data
yp = useBayes(XtrC, modelC)

# Índices dentro de ytrC (e yp), que tem só as amostras de treino
ix1 = np.where(ytrC == 1)[0]
ix2 = np.where(ytrC == 2)[0]

TP = np.sum(yp[ix1] == 1)
FN = np.sum(yp[ix1] != 1)
TN = np.sum(yp[ix2] == 2)
FP = np.sum(yp[ix2] != 2)

SS = TP/(TP+FN)
SP = TN/(TN+FP)
PR = TP/(TP+FP)
F1 = 2*PR*SS/(PR+SS)
AC = (TN+TP)/(TP+TN+FP+FN)

print("Sensitivity(%)=" + str(SS*100))
print("Specificity(%)=" + str(SP*100))
print("Precision(%)=" + str(PR*100))
print("F1Score(%)=" + str(F1*100))
print("Accuracy(%)=" + str(AC*100))

#plot the decision plane
plotDecision(XtrC, ytrC, modelC, granularity=2, classContours=False, f1name="ART", f2name="PRT")

#Plot conditional PDFs as 3D surfaces
xp = np.arange(30, 500, 5)
yp_grid = np.arange(100, 1000, 5)
Xp, Yp = np.meshgrid(xp, yp_grid)
XX = np.array([Xp.ravel(), Yp.ravel()]).T

Z1 = pdfGauss(XX, modelC['mean1'], modelC['cov1']).reshape(Xp.shape)
Z2 = pdfGauss(XX, modelC['mean2'], modelC['cov2']).reshape(Xp.shape)

fig = go.Figure()
fig.add_trace(go.Surface(x=Xp, y=Yp, z=Z1))
fig.add_trace(go.Surface(x=Xp, y=Yp, z=Z2))
fig.update_traces(contours_z=dict(show=True, usecolormap=True,
                                  highlightcolor="limegreen", project_z=True))
fig.update_scenes(xaxis_title_text='ART', yaxis_title_text='PRT', zaxis_title_text='PDF')
fig.update_layout(autosize=False, width=1000, height=1000)
fig.show()

Dimensões do dataset: (150, 11)
Features: ['ART', 'N.1', 'PRT', 'ARM', 'PRM', 'ARTG', 'NG', 'PRTG', 'RAAR', 'RAN']
Número de amostras da classe 1: 50
Número de amostras da classe 2: 50


c:\Users\josem\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\josem\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\josem\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\josem\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^

Sensitivity(%)=92.0
Specificity(%)=84.0
Precision(%)=85.18518518518519
F1Score(%)=88.46153846153845
Accuracy(%)=88.0


c:\Users\josem\anaconda3\Lib\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.




In [2]:
#Fit bayes for the training data

def pdfGauss(X,mean,cov):
    covInv=np.linalg.inv(cov)
    dim=cov.shape[0]
    val=np.array([])
    for i in range(X.shape[0]):
        dist=((np.array([X[i,:]-mean]))@covInv@(np.array([X[i,:]-mean])).T).squeeze()
        val=np.append(val,np.exp(-0.5*dist)/((2*np.pi)**(dim/2)*np.linalg.det(cov)**0.5))
    return np.array([val]).T


def fitBayes3K(Xtr,ytr):
    ix1=np.where(ytr==1)[0]
    ix2=np.where(ytr==2)[0]
    ix3=np.where(ytr==3)[0]
    
    #Compute priors
    Pw1=ix1.shape[0]/(ix1.shape[0]+ix2.shape[0]+ix3.shape[0])
    Pw2=ix2.shape[0]/(ix1.shape[0]+ix2.shape[0]+ix3.shape[0])
    Pw3=ix3.shape[0]/(ix1.shape[0]+ix2.shape[0]+ix3.shape[0])
    
    # Estimate gaussian conditional PDFs --> fit three Gaussian Mixture Models
    clf1 = mixture.GaussianMixture(n_components=1)
    clf2 = mixture.GaussianMixture(n_components=1)
    clf3 = mixture.GaussianMixture(n_components=1)
    mod1=clf1.fit(Xtr[ix1,:])
    mod2=clf2.fit(Xtr[ix2,:])
    mod3=clf3.fit(Xtr[ix3,:])
    
    return {'mean1':mod1.means_.squeeze(),'mean2':mod2.means_.squeeze(),'mean3':mod3.means_.squeeze(),'cov1':mod1.covariances_[0],
            'cov2':mod2.covariances_[0],'cov3':mod3.covariances_[0],'Pw1':Pw1,'Pw2':Pw2,'Pw3':Pw3}

#Apply a trained bayes classifier
def useBayes3K(Xte,model):
    Pw1X = pdfGauss(Xte,model['mean1'],model['cov1'])*model['Pw1']
    Pw2X = pdfGauss(Xte,model['mean2'],model['cov2'])*model['Pw2']
    Pw3X = pdfGauss(Xte,model['mean3'],model['cov3'])*model['Pw3']
    
    PwXX=np.concatenate((Pw1X,Pw2X,Pw3X),axis=1)
    
    res=np.array([])
    for i in range(PwXX.shape[0]):
        res=np.append(res,np.where(PwXX[i,:]==np.max(PwXX[i,:]))[0]+1)
    
    return res


#Features
ART=np.array([CorkStopp['ART'].to_numpy()])

PRM=np.array([CorkStopp['PRM'].to_numpy()])

PRT=np.array([CorkStopp['PRT'].to_numpy()])


#Training Set
XtrC2=np.concatenate((ART,PRM,PRT),axis=0).T
ytrC2=CorkStopp['C'].to_numpy()


modelC2=fitBayes3K(XtrC2,ytrC2)

yp=useBayes3K(XtrC2,modelC2)

Hits=np.shape(np.where((ytrC2==yp))[0])[0]
AC=Hits/XtrC2.shape[0]

print("Error(%)="+str(100-(AC)*100))


Error(%)=9.333333333333343


c:\Users\josem\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.

c:\Users\josem\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.

c:\Users\josem\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.

